In [0]:
# Imports

from pyspark.sql.functions import col, trim, to_date, expr, when, current_timestamp
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
from pyspark.sql.functions import col

In [0]:
# Ler a Bronze V2
df_bronze = spark.table("workspace.drs_bronze.estabelecimentos_saude_goias")

In [0]:
# Padronizando os nomes das colunas

df_estabelecimento_saude = df_bronze \
    .withColumnRenamed("cnes", "cnes_id") \
    .withColumnRenamed("razao_social", "legal_name") \
    .withColumnRenamed("nome_fantasia", "trade_name") \
    .withColumnRenamed("logradouro", "address_street") \
    .withColumnRenamed("numero", "address_number") \
    .withColumnRenamed("complemento", "address_complement") \
    .withColumnRenamed("bairro", "neighborhood") \
    .withColumnRenamed("cep", "zip_code") \
    .withColumnRenamed("telefone", "phone_number") \
    .withColumnRenamed("email", "email_address") \
    .withColumnRenamed("url", "website_url") \
    .withColumnRenamed("latitude", "latitude") \
    .withColumnRenamed("longitude", "longitude") \
    .withColumnRenamed("codigo_ibge", "ibge_city_code") \
    .withColumnRenamed("uf", "state_abbreviation") \
    .withColumnRenamed("municipio", "city_name") \
    .withColumnRenamed("tipo_gestao", "management_type") \
    .withColumnRenamed("natureza_juridica", "legal_nature") \
    .withColumnRenamed("vinculo_sus", "sus_affiliation") \
    .withColumnRenamed("tipo_estabelecimento", "establishment_type") \
    .withColumnRenamed("ingestion_timestamp", "ingestion_timestamp") \
    .withColumnRenamed("source_file", "source_filename")

print(f"Total registros: {df_estabelecimento_saude.count()}")


In [0]:
# Transformar/Convert colunas para o padarão para depois salvar a Silver V2

from pyspark.sql.functions import col, trim, to_date, expr, when

# Registra o DataFrame como uma tabela chamada 'v_saude'
df_estabelecimento_saude.createOrReplaceTempView("v_saude")

df_silver_casting = spark.sql("""
    WITH df_silver_with_out_address_number_sn AS(
        -- Remover linhas com 's/n' na coluna 'address_number
        SELECT 
            cnes_id,
            CASE 
                WHEN address_number = 's/n' THEN NULL 
                ELSE address_number 
            END as address_number_cleaned
        FROM v_saude
    ),
    df_silver_with_out_address AS (
        SELECT  cnes_id,
            legal_name,
            trade_name,
            address_street,
            address_complement,
            neighborhood,
            zip_code,
            phone_number,
            email_address,
            website_url,
            latitude,
            longitude,
            ibge_city_code,
            state_abbreviation,
            city_name,
            management_type,
            legal_nature,
            sus_affiliation,
            establishment_type,
            ingestion_timestamp,
            source_filename
        FROM v_saude
    )
    -- join df_silver_with_out_address_number_sn and df_silver_with_out_address
    
    SELECT 
        b.*,
        a.* EXCEPT (address_number_cleaned,cnes_id), -- remove field address_number_cleaned
        a.address_number_cleaned as address_number
        from df_silver_with_out_address_number_sn as a
        inner join df_silver_with_out_address as b
        on a.cnes_id = b.cnes_id      

""")
df_silver = (
    df_silver_casting
    .withColumn("address_number", col("address_number").cast("integer")) 
)

In [0]:
# criando o campo source_file com lit
from pyspark.sql.functions import lit

df = df_silver.withColumn("source_file", lit("estabelecimentos_saude_goias.csv"))

In [0]:
# Separando registros inválidos para quarentena

df_invalid = df.filter("""
address_number is null
""")

print(f"Total registros inválidos: {df_invalid.count()}")

In [0]:
%python

#Salvando quarentena da Silver V2 and  Drop cnes_id if it already exists in the target table

# if "cnes_id" in df_invalid.columns:
#     df_invalid = df_invalid.drop("cnes_id")

df_invalid.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("workspace.drs_silver.estabelecimento_saude_go_quarantine")


In [0]:
# criando o campo ingestion_timestamp
from pyspark.sql.functions import current_timestamp

df_silver = df.withColumn("ingestion_timestamp", current_timestamp())

In [0]:
# # Filtrando valores válidos
# from pyspark.sql.functions import col

# # df_silver_filtered = (
#     df_silver.dropDuplicates()
#       .filter(
#           (col("cnes_id").isNotNull()) &
#           (col("legal_name").isNotNull()) &
#           (col("trade_name").isNotNull()) &
#           (col("address_street").isNotNull()) &
#           (col("address_number").isNotNull()) 
#       )
# )


In [0]:
# Criando o campo silver_processed_timestamp em df_silver
from pyspark.sql.functions import current_timestamp

df_silver = df_silver \
.withColumn("silver_processed_timestamp", current_timestamp()) \
.withColumn("created_at", current_timestamp()) \
.withColumn("updated_at", current_timestamp())

In [0]:
%python 
# Grantindo a deduplicação por chave de negócio
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number

window_spec = Window.partitionBy(
    "cnes_id",
    "ingestion_timestamp"
    ).orderBy(
    col("ingestion_timestamp").desc()
)

df_silver = (
    df_silver
    .withColumn("row_num", row_number().over(window_spec))
    .filter(col("row_num") == 1)
    .drop("row_num")
)


In [0]:
# Salvando a tabela Silver em uma staging table

df_silver.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema", "true") \
.saveAsTable("workspace.drs_silver.estabelecimentos_saude_go_staging")

In [0]:

# Craindo tabela final se não existir

spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.drs_silver.estabelecimentos_saude_go
AS
SELECT * FROM workspace.drs_silver.estabelecimentos_saude_go_staging WHERE 1 = 0
""")

In [0]:
# Validar schema da staging

spark.table("workspace.drs_silver.estabelecimentos_saude_go_staging").printSchema()

In [0]:
# Salvando tabela drs_silver_v2 com MERGE

spark.sql("""
MERGE INTO workspace.drs_silver.estabelecimentos_saude_go AS target
USING workspace.drs_silver.estabelecimentos_saude_go_staging AS source

ON target.cnes_id = source.cnes_id

WHEN MATCHED THEN
UPDATE SET
    target.legal_name = source.legal_name,
    target.trade_name = source.trade_name,
    target.address_street = source.address_street,
    target.address_number = source.address_number,
    target.address_complement = source.address_complement,
    target.neighborhood = source.neighborhood,
    target.zip_code = source.zip_code,
    target.phone_number = source.phone_number,
    target.email_address = source.email_address,
    target.website_url = source.website_url,
    target.latitude = source.latitude,
    target.longitude = source.longitude,
    target.ibge_city_code = source.ibge_city_code,
    target.state_abbreviation = source.state_abbreviation,
    target.city_name = source.city_name,
    target.management_type = source.management_type,
    target.legal_nature = source.legal_nature,
    target.sus_affiliation = source.sus_affiliation,
    target.establishment_type = source.establishment_type,
    target.ingestion_timestamp = source.ingestion_timestamp,
    target.source_filename = source.source_filename,
    target.source_file = source.source_file,
    target.silver_processed_timestamp = source.silver_processed_timestamp,
    target.updated_at = current_timestamp()

WHEN NOT MATCHED THEN
INSERT (
    cnes_id,
    legal_name,
    trade_name,
    address_street,
    address_number,
    address_complement,
    neighborhood,
    zip_code,
    phone_number,
    email_address,
    website_url,
    latitude,
    longitude,
    ibge_city_code,
    state_abbreviation,
    city_name,
    management_type,
    legal_nature,
    sus_affiliation,
    establishment_type,
    ingestion_timestamp,
    source_filename,
    source_file,
    silver_processed_timestamp,
    created_at,
    updated_at
)
VALUES (
    source.cnes_id,
    source.legal_name,
    source.trade_name,
    source.address_street,
    source.address_number,
    source.address_complement,
    source.neighborhood,
    source.zip_code,
    source.phone_number,
    source.email_address,
    source.website_url,
    source.latitude,
    source.longitude,
    source.ibge_city_code,
    source.state_abbreviation,
    source.city_name,
    source.management_type,
    source.legal_nature,
    source.sus_affiliation,
    source.establishment_type,
    source.ingestion_timestamp,
    source.source_filename,
    source.source_file,
    source.silver_processed_timestamp,
    current_timestamp(),
    current_timestamp()
)
""")


In [0]:
# Validando a tabela drs_silver_votacao_secao_2024_go

spark.sql("""
SELECT
* 
FROM workspace.drs_silver.estabelecimentos_saude_go
""")

In [0]:
# Verificar duplicidade pela chave do merge

spark.sql("""
SELECT
    cnes_id,
    ingestion_timestamp,
    COUNT(*) AS qtd
FROM workspace.drs_silver.estabelecimentos_saude_go
GROUP BY cnes_id,ingestion_timestamp 
HAVING COUNT(*) > 1
ORDER BY qtd DESC
""").display()

In [0]:
# Data Quality checks da Silver V2
# dq = spark.sql("""
# SELECT
#     SUM(CASE WHEN cnes_id IS NULL THEN 1 ELSE 0 END) AS cnes_id
# FROM workspace.drs_silver.estabelecimento_saude_go_quarantine
# """)

# display(dq)